# 03 · النشر والأرشيف
يشغّل الخط كامل مثل GitHub Actions بالضبط، ويكتب `site/data/showtimes.json`.
وتحت تحليل بسيط للأرشيف التاريخي، وهو بداية فكرة تقارير البيانات للشركات.

In [ ]:
import sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
import pandas as pd
pd.set_option("display.max_rows", 60)

In [ ]:
from pipeline.run import run
payload = run()   # للتجربة بدون إنترنت: run(fixture=ROOT/"tests/fixtures/vox_showtimes.html")
{k: len(v) if isinstance(v, (list, dict)) else v for k, v in payload.items()}

In [ ]:
pd.DataFrame(payload["sources"]).T

## الأرشيف
كل عرض نشوفه يُحفظ مرة وحدة في `data/history/shows_YYYY-MM.csv`.

In [ ]:
files = sorted((ROOT / "data" / "history").glob("shows_*.csv"))
hist = pd.concat([pd.read_csv(f) for f in files]) if files else pd.DataFrame()
print(len(hist), "عرض في الأرشيف")
if not hist.empty:
    hist["start"] = pd.to_datetime(hist["start"])
    display(hist.groupby(["city","title_clean"]).size().rename("عدد العروض")
                .sort_values(ascending=False).head(15))

In [ ]:
# حصة كل فلم من العروض لكل يوم: هذا نوع الأرقام اللي تهم الموزعين
if not hist.empty:
    daily = (hist.assign(day=hist["start"].dt.date)
                 .groupby(["day","title_clean"]).size().unstack(fill_value=0))
    share = daily.div(daily.sum(axis=1), axis=0).round(3)
    display(share.tail(7).T.sort_values(share.index[-1], ascending=False).head(10))